# Evaluate v4 on held-out in-domain test setsRuns the v4 adapter against the held-out test splits of the 5 gold corpora and reports component-F1. Produces the v4 column in Appendix B of the paper.

In [ ]:
#inlcuding aaec

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
import sys, json, time
sys.path.insert(0, '.')
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student
from src.phase2.dataset import read_jsonl
from src.phase2.evaluate import evaluate_corpus

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len = 2048
cfg.student.max_target_len = 2048   # bumped for v4's CoT format

student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print(f"loaded v4 with max_target={cfg.student.max_target_len}", flush=True)

print("\n=== v4 in-domain eval with CoT-sized budget ===\n", flush=True)
CAP = 50   # keep cap for speed since each row now takes ~5 sec
summary = {}
for dom in ('microtext', 'cdcp', 'abstrct', 'perspectrum', 'aaec'):
    recs = read_jsonl(f'phase2_data/unified/{dom}_test.jsonl')[:CAP]
    if not recs: continue
    t0 = time.time()
    preds = []
    for i, r in enumerate(recs, 1):
        try:
            pred, _ = student.predict(r['input'])
            preds.append(pred)
        except Exception as e:
            preds.append({'claim_components':[],'premise_components':[],
                          'citation_components':[],'relations':[]})
        if i % 25 == 0:
            print(f"  {dom} {i}/{len(recs)} ({time.time()-t0:.0f}s)", flush=True)
    m = evaluate_corpus(recs, preds, threshold=0.5)
    empty = sum(1 for p in preds
                if len(p['claim_components'])==0
                and len(p['premise_components'])==0)
    print(f"  ── {dom} ── n={len(recs)} empty={empty} ({100*empty/len(recs):.0f}%)")
    print(f"     comp-F1={m['macro_component_f1']:.3f}")
    summary[dom] = m['macro_component_f1']

print("\n=== SUMMARY (v4 with max_target=2048, cap 50) ===")
for dom, f1 in summary.items():
    print(f"  {dom:14s} comp-F1={f1:.3f}")
avg = sum(summary.values())/len(summary)
print(f"  {'average':14s} comp-F1={avg:.3f}")
print(f"\n  vs v3 (uncapped, no CoT budget): 0.207 avg")
print(f"  vs Phase 2-α (capped 50):         0.333 avg")
PY